# Inspect Pruning Results

Load combined results from a sweep run and explore accuracy vs. sparsity.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Set this to the run directory you want to inspect.
RUN_DIR = Path("../outputs/runs")  # adjust as needed

In [ ]:
# Find the most recent run directory that has a combined_results.csv.
candidates = sorted(
    [p for p in RUN_DIR.iterdir() if (p / "combined_results.csv").exists()],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if not candidates:
    raise FileNotFoundError(f"No completed runs found under {RUN_DIR}")
run_dir = candidates[0]
print("Loading from:", run_dir)

In [ ]:
df = pd.read_csv(run_dir / "combined_results.csv")
df = df.sort_values(["model_name", "sparsity_requested"])
df[["model_name", "sparsity_requested", "sparsity_achieved", "accuracy", "num_samples"]].head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for model_name, group in df.groupby("model_name"):
    ax.plot(group["sparsity_requested"], group["accuracy"], marker="o", label=model_name)
ax.set_xlabel("Sparsity (%)")
ax.set_ylabel("MMLU Accuracy")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Per-subject accuracy for a selected model and sparsity.
MODEL = df["model_name"].iloc[0]  # change as needed
SPARSITY = 0

metrics_path = run_dir / MODEL / f"sparsity_{int(SPARSITY):03d}" / "metrics.json"
with metrics_path.open() as f:
    m = json.load(f)

per_subject = pd.Series(m["per_subject_accuracy"]).sort_values()
per_subject.plot.barh(figsize=(8, max(4, len(per_subject) * 0.3)))
plt.xlabel("Accuracy")
plt.title(f"{MODEL} — sparsity {SPARSITY}%")
plt.tight_layout()
plt.show()

In [ ]:
# Manifest summary.
with (run_dir / "manifest.json").open() as f:
    manifest = json.load(f)

print(f"Run name   : {manifest['run_name']}")
print(f"Start time : {manifest['start_time']}")
print(f"End time   : {manifest['end_time']}")
print(f"Completed  : {len(manifest['completed'])}")
print(f"Skipped    : {len(manifest['skipped'])}")
print(f"Failed     : {len(manifest['failed'])}")
if manifest["failed"]:
    print("\nFailures:")
    for entry in manifest["failed"]:
        print(f"  {entry}")